# 02 · Lineup Score para predicción W/L

Este notebook construye la feature **LINEUP_SCORE_DIFF** combinando quality individual (on/off) y sinergia histórica de alineaciones.\n
- Inputs: dashboards de lineups, on/off por jugador y boxscore tradicional.\n
- Importante: asegurar dtypes (TEAM_ID/PLAYER_ID int64, GAME_ID string) y respetar el formato `-p1-p2-p3-p4-p5-` de `GROUP_ID`.\n
- El flujo evita fuga temporal trabajando con información post-partido.


In [ ]:
# Parámetros y rutas
from pathlib import Path
import pandas as pd
import numpy as np
import importlib.util

PARAMS = dict(
    m0_player_minutes=400,
    M0_lineup_minutes=300,
    alpha_quality=0.75,
    winsor_limits=(0.05, 0.95),
)
PATHS = dict(
    lineups="/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_dash_lineups__dataset_1.parquet",
    on="/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_1.parquet",
    off="/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_2.parquet",
    boxscore="/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/boxscores/boxscore_traditional.parquet",
    out="/Users/pablo/Documents/BigData/BasketballAnalysis/02_processing_data/02a_WL_prediction/features/lineup_scores.parquet",
)

def _load_feature_module():
    module_path = Path('02_processing_data/02a_WL_prediction/02_FeatureFunctions.py').resolve()
    spec = importlib.util.spec_from_file_location('feature_functions', module_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

feature_mod = _load_feature_module()
print('Funciones de features cargadas desde', feature_mod.__file__)


In [ ]:
# Carga & saneo
dfs = {}
for key in ['lineups', 'on', 'off', 'boxscore']:
    path = Path(PATHS[key])
    if not path.exists():
        print(f'⚠️ No encuentro {key}: {path}')
        continue
    dfs[key] = pd.read_parquet(path)
    if 'game_id' in dfs[key].columns:
        dfs[key] = dfs[key].drop(columns=['game_id'])
    if 'GAME_ID' in dfs[key].columns:
        dfs[key]['GAME_ID'] = dfs[key]['GAME_ID'].astype('string')
    for col in ['TEAM_ID', 'PLAYER_ID']:
        if col in dfs[key].columns:
            dfs[key][col] = pd.to_numeric(dfs[key][col], errors='coerce').astype('Int64')
    print(f"{key} -> shape {dfs[key].shape}")
    display(dfs[key].dtypes)
    display(dfs[key].head())

df_box = dfs.get('boxscore', pd.DataFrame())
df_lineups = dfs.get('lineups', pd.DataFrame())
df_on = dfs.get('on', pd.DataFrame())
df_off = dfs.get('off', pd.DataFrame())


In [ ]:
# Construcción de alineaciones por partido
if not df_box.empty:
    df_game_lineups = feature_mod.extract_game_lineups_from_boxscore(df_box)
    print('Alineaciones detectadas:', len(df_game_lineups))
    display(df_game_lineups.sample(min(5, len(df_game_lineups))))
else:
    print('Sin boxscore -> no puedo construir alineaciones.')
    df_game_lineups = pd.DataFrame()


In [ ]:
# Quality individual (On/Off)
if not df_on.empty and not df_off.empty:
    # Compatibilidad: los datasets on/off usan VS_PLAYER_ID en lugar de PLAYER_ID
    if 'VS_PLAYER_ID' in df_on.columns and 'PLAYER_ID' not in df_on.columns:
        print('Renombro VS_PLAYER_ID -> PLAYER_ID en df_on para calcular quality.')
        df_on = df_on.rename(columns={'VS_PLAYER_ID': 'PLAYER_ID'})
    if 'VS_PLAYER_ID' in df_off.columns and 'PLAYER_ID' not in df_off.columns:
        print('Renombro VS_PLAYER_ID -> PLAYER_ID en df_off para calcular quality.')
        df_off = df_off.rename(columns={'VS_PLAYER_ID': 'PLAYER_ID'})
    df_quality = feature_mod.compute_player_quality_from_onoff(
        df_on, df_off,
        m0_player_minutes=PARAMS['m0_player_minutes'],
        winsor_limits=PARAMS['winsor_limits'],
    )
    display(df_quality.head())
    _vals = df_quality['quality_player'].dropna()
    if not _vals.empty:
        ax = _vals.hist(bins=30, figsize=(6, 4))
        ax.set_title('Distribución quality_player')
        display(ax.figure)
        clip_lo, clip_hi = _vals.quantile(PARAMS['winsor_limits'])
        outliers = df_quality[(df_quality['quality_player'] < clip_lo) | (df_quality['quality_player'] > clip_hi)]
        print('Outliers winsorizados (preview):', len(outliers))
        display(outliers.head())
else:
    print('Sin on/off -> no se calcula quality.')
    df_quality = pd.DataFrame()


In [ ]:
# Synergy por GROUP_ID
if not df_lineups.empty:
    df_synergy = feature_mod.compute_lineup_synergy_from_dashboard(
        df_lineups,
        M0_lineup_minutes=PARAMS['M0_lineup_minutes'],
    )
    top_team = df_synergy.groupby('TEAM_ID')['minutes_lineup'].sum().idxmax() if not df_synergy.empty else None
    if top_team is not None:
        display(df_synergy[df_synergy['TEAM_ID'] == top_team].sort_values('minutes_lineup', ascending=False).head(10))
    else:
        display(df_synergy.head())
else:
    print('Sin dashboard de lineups -> no hay synergy.')
    df_synergy = pd.DataFrame()


In [ ]:
# Scoring por partido
if not df_game_lineups.empty and not df_quality.empty and not df_synergy.empty:
    df_scores = feature_mod.build_lineup_scores_for_games(
        df_box=df_box,
        df_lineups=df_lineups,
        df_on=df_on,
        df_off=df_off,
        params=PARAMS,
    )
    display(df_scores.head())
else:
    print('Faltan insumos -> df_scores vacío.')
    df_scores = pd.DataFrame()


In [ ]:
# Validación exploratoria (rápida)
if not df_scores.empty:
    if 'WL_NUM' in df_box.columns:
        df_wl = df_box[['GAME_ID', 'TEAM_ID', 'WL_NUM', 'MATCHUP']].drop_duplicates()
        df_home = df_wl[df_wl['MATCHUP'].str.contains('vs', case=False, na=False)][['GAME_ID', 'WL_NUM']].rename(columns={'WL_NUM': 'WL_HOME'})
        df_val = df_scores.merge(df_home, on='GAME_ID', how='left')
        if df_val['WL_HOME'].notna().any():
            corr_pearson = df_val[['WL_HOME', 'LINEUP_SCORE_DIFF']].corr(method='pearson').iloc[0, 1]
            corr_spearman = df_val[['WL_HOME', 'LINEUP_SCORE_DIFF']].corr(method='spearman').iloc[0, 1]
            print(f'Pearson(WL_HOME, diff) = {corr_pearson:.3f}')
            print(f'Spearman(WL_HOME, diff) = {corr_spearman:.3f}')
        else:
            print('Sin WL_HOME -> omito correlaciones.')
    else:
        print('Boxscore sin WL_NUM -> omito correlaciones.')
    ax = df_scores['LINEUP_SCORE_DIFF'].hist(bins=30, figsize=(6, 4))
    ax.set_title('Distribución LINEUP_SCORE_DIFF')
    display(ax.figure)
else:
    print('Sin df_scores -> omito validación.')


In [ ]:
# Export
if not df_scores.empty:
    out_path = Path(PATHS['out'])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_scores.to_parquet(out_path, index=False)
    print(f'Guardado OK -> {out_path} ({df_scores.shape[0]} filas, {df_scores.shape[1]} columnas)')
else:
    print('df_scores vacío -> no se exporta.')


## Notas de predicción pre-partido

Para usar este bloque en predicciones previas al juego, filtra los dataframes de entrada por `GAME_DATE` < fecha del partido objetivo antes de calcular quality y synergy. Mantén el formato canónico de `GROUP_ID` y evita mezclar minutos futuros para prevenir fuga de información.
